In [ ]:
# ======================
# IMPORT
# ======================
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import json

# ======================
# 1. LOAD DATA
# ======================
df = pd.read_excel(r"D:\Dashboard\superstore_mba_augmented (1).xlsx")  # sửa lại path nếu cần

# ======================
# 2. PREPROCESS
# ======================
basket = df.groupby(['Order ID', 'Sub-Category'])['Quantity'] \
           .sum().unstack().fillna(0)

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

# ======================
# 3. ASSOCIATION RULES
# ======================
frequent_itemsets = apriori(
    basket,
    min_support=0.005,
    use_colnames=True
)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.03
)

# ======================
# 4. TOP 2 RULES / SUBCATEGORY
# ======================
def get_top_rules_per_subcategory(rules_df):
    result = {}

    for sub in basket.columns:
        sub_rules = rules_df[
            rules_df['antecedents'].apply(lambda x: sub in x) |
            rules_df['consequents'].apply(lambda x: sub in x)
        ]

        sub_rules = sub_rules.sort_values(by="confidence", ascending=False)

        clean_rules = []

        for _, row in sub_rules.iterrows():
            antecedent = list(row['antecedents'])
            consequent = list(row['consequents'])

            if sub in antecedent:
                targets = [x for x in consequent if x != sub]
            else:
                targets = [x for x in antecedent if x != sub]

            if targets:
                clean_rules.append({
                    "target_subcategory": targets[0],
                    "confidence": float(row['confidence'])
                })

        # remove duplicate + lấy top 2
        seen = set()
        unique_rules = []

        for r in clean_rules:
            if r['target_subcategory'] not in seen:
                unique_rules.append(r)
                seen.add(r['target_subcategory'])

            if len(unique_rules) == 1:
                break

        result[sub] = unique_rules

    return result

top_rules = get_top_rules_per_subcategory(rules)

# ======================
# 5. TOP 5 PRODUCTS
# ======================
top_products_df = (
    df.groupby(['Sub-Category', 'Product Name'])['Quantity']
    .sum()
    .reset_index()
    .sort_values(['Sub-Category', 'Quantity'], ascending=[True, False])
)

top_products = {}

for sub in df['Sub-Category'].unique():
    sub_df = top_products_df[top_products_df['Sub-Category'] == sub]
    top_products[sub] = sub_df.head(5)['Product Name'].tolist()

# ======================
# 6. BUILD JSON
# ======================
output = {"subcategories": {}}

for sub in basket.columns:
    output["subcategories"][sub] = {
        "top_products": top_products.get(sub, []),
        "rules": top_rules.get(sub, [])
    }

# ======================
# 7. EXPORT JSON
# ======================
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ data.json đã tạo xong")

✅ data.json đã tạo xong


C:\Users\ROG\AppData\Local\Temp\ipykernel_27580\3843776637.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)
c:\Users\ROG\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [2]:
rules_output = []

for _, row in rules.iterrows():
    rules_output.append({
        "antecedents": list(row['antecedents']),
        "consequents": list(row['consequents']),
        "confidence": float(row['confidence'])
    })

output = {
    "subcategories": output["subcategories"],
    "rules": rules_output
}